# logit-thermo A2-27：A2-19 の罰なしの腕の文を、同じモデルにもう一度通して採点し直す（GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。
- **ランタイム：L4**（ハイメモリはいりません）。GPU のメモリが 20GB に足りなければ、⚠ を送ってランタイムを切ります。
- **生成はしません。** A2-19 の罰なしの文（Drive の `A2_19/`）をモデルに 1 回通して、歩ごとの量（エネルギーの見込みなど）を保存するだけです。仕事は 21 個（小さい 7 本 × 記事の 3 組）。1 つ 3〜5 分くらいで、1〜1.5 時間、CU 2〜3 の見込みです。
- 早く終わらせたいときは、このノートブックを 2 つ目のランタイムでも「すべて実行」してください。仕事は取りに行く方式なので重なりません。
- 保存先 `A2_27/` と `A2_27/locks/` は、Claude が先に Drive に作ってあります。
- **最初に、出し直した U と q0 が、生成のときに保存したものと合うかを仕事ごとに確かめます。** 合わなければ ⚠ を送って止まります。
- **データは、分析のコードを回すまで開けません。** 結果も記事の中身も表示しません。途中で切れても、もう一度「すべて実行」すれば、終わった仕事は飛ばします。
- **通知**：🚀・仕事ごとの ✅・🎉・⚠️。名札は「logit-thermo A2-27（A2-19 の罰なしの文を小さい 7 本で採点し直す、仕事 21 個、L4）」。終わったら、またはどのセルでもエラーで止まったら、ランタイムを切ります（手で止めたときも切れます）。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-27（A2-19 の罰なしの文を小さい 7 本で採点し直す、仕事 21 個、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-27 rescoring of A2-19 c0 arm', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
def _cut_on_error(result):
    # どのセルでエラーになっても（A2-19 ではセル 2 で止まったときに切れなかった）、⚠ を送ったあとでランタイムを切る。手で止めたときも切れます
    if result.error_in_exec is not None and globals().get('AUTO_DISCONNECT', True):
        time.sleep(10)
        try:
            from google.colab import drive as _d; _d.flush_and_unmount()
        except Exception:
            pass
        from google.colab import runtime as _r; _r.unassign()
if not globals().get('_CUT_ON'):
    get_ipython().events.register('post_run_cell', _cut_on_error); _CUT_ON = True

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル1b：このノートブックの中身が、ゲート（docs/A2_27_GATE.md）に書いたハッシュと合うかを確かめる（Drive に上げるときの写しまちがいを見つけるため）
import glob, hashlib, json
_hits = sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/A2_27_L4.ipynb', recursive=True))
assert len(_hits) == 1, ('Drive に A2_27_L4.ipynb が 1 つだけ見つかりません', _hits)
_cells = json.load(open(_hits[0]))['cells']
_src = lambda c: c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
_want = {0: '9ae300fd4765bde995f6bf6e0a29c3b9d3f9a9d4eab959604cfe1740a030468b', 1: 'bd906a526ee7894be076fcc40fed64b8311e4974fe549419344223ba9da619c7', 2: 'd5c45a43b6d0062568255716bb9b376c6b05a0402d7520e8baf115e338440021', 3: 'e394b92364d4be993da910864638aee600422ef5dec1b2f5b5bcb619a02b9c37', 4: 'b32ae85c34e28b22442c33b8613dd7c407cb8e3ce9dfdee4d9f383fb3ab94ce8'}
_got = {i: hashlib.sha256(_src(_cells[i if i < 2 else i + 1]).encode()).hexdigest() for i in _want}
_bad = [i for i in _want if _got[i] != _want[i]]
assert not _bad, ('ノートブックの中身がゲートのハッシュと違います（Drive に上げるときの写しまちがい）。回さずに Claude に知らせてください', _bad)
print('ノートブックの中身はゲートのハッシュと一致しました')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', 'SRC7BC': 'b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0', 'SRCF': '0ac42c7ab82b06d2e7443ba9c0c0e147c0121b5351e0b5830b1a54b46043190d', 'SRCD': '33f47a0a3bbda18341205dedb94c6d200f295fc4136f5cb8e0ee067b2d7f518c', 'SRCD2': 'cddae786903604183e46708a6606b482f720bba0b0df0a8df724440c9bde51b2', 'SRCG': '49aa63e0c3479f9841bf4b17740325fe21ddaea8b118f03730461959ef03e44b'})
SRC22, SRC24, SRC7C, SRC7BC, SRCF, SRCD = _s['SRC22'], _s['SRC24'], _s['SRC7C'], _s['SRC7BC'], _s['SRCF'], _s['SRCD']
SRCD, SRCD2 = _s['SRCD'], _s['SRCD2']   # 名前を 2 つ並べて置く（take_src が読み飛ばす形）
SRCD2, SRCG = _s['SRCD2'], _s['SRCG']   # 名前を 2 つ並べて置く（take_src が読み飛ばす形）
def take_file(name, want):
    """Drive の logit-thermo のフォルダから、名前とハッシュが合う .py のファイルを読む。"""
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/' + name, recursive=True)):
        s = open(p, encoding='utf-8').read()
        if hashlib.sha256(s.encode()).hexdigest() == want:
            return s
    raise AssertionError(('Drive に ' + name + ' （ハッシュが合うもの）が見つかりません', want))
SRC17 = take_file('a2_18_collect.py', '0b9d203ebe0a71e67d3c419897212c2091f04697c0187ac9475f9112eeea6496')
SRC19 = take_file('a2_19_collect.py', 'de4d86520b5ab6a42f02a7dea8a3048a943711ba26499d3a08ca061aad27b3ef')
SRC20 = take_file('a2_20_collect.py', '29d5dbd7a05620433f5bf0d1ad9dc3c9e130f89d8ed66622139149d51fea99d6')
SRC27 = take_file('a2_27_collect.py', 'd3ebb9da9afaeeeb05efa3fd83ab714ad9cdb4d5d12d85019612e68a96bd57d5')
exec(SRC22)
exec(SRC24)
exec(SRC7C)
exec(SRC7BC)
exec(SRCF)
exec(SRCD)
exec(SRCD2)
exec(SRCG)
exec(SRC17)
exec(SRC19)
exec(SRC20)
exec(SRC27)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → BBC の記事（A2-2・A2-19 と同じか目録で確かめる）→ 仕事を取りに行って採点し直す → 保存（結果も記事も表示しない）
import json, sys, transformers
try:
    torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
    print('自己テスト：', self_test_rs27('cuda'), self_test_dsc2('cuda'), self_test_gpre(), self_test_a18(), self_test_a19(), self_test_a20())
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    GROUPS, META19 = load_docs19('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json')
    check_docs20(META19, json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_19/docs_meta_w0.json')), 'groups')
    assert os.path.isdir('/content/drive/MyDrive/logit-thermo/A2_27') and os.path.isdir('/content/drive/MyDrive/logit-thermo/A2_27' + '/locks'), ('Claude が先に作った保存先のフォルダが見つかりません', '/content/drive/MyDrive/logit-thermo/A2_27')
    NAMES = list(A18_HF)
    assert len(NAMES) == 7, NAMES
    N_DONE, LEFT = rs27_run(NAMES, A18_HF, 0, 1, GROUPS, '/content/drive/MyDrive/logit-thermo/A2_19', '/content/drive/MyDrive/logit-thermo/A2_27', 'cuda', notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(done=N_DONE, left=LEFT, batch=RS27_B, a27_sha256='d3ebb9da9afaeeeb05efa3fd83ab714ad9cdb4d5d12d85019612e68a96bd57d5', a19_sha256='de4d86520b5ab6a42f02a7dea8a3048a943711ba26499d3a08ca061aad27b3ef',
                   torch=torch.__version__, transformers=transformers.__version__, python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0),
                   matmul=torch.get_float32_matmul_precision()),
              open('/content/drive/MyDrive/logit-thermo/A2_27' + '/run_meta_' + A19_TOKEN[:8] + '.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-27（採点し直し）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify(f'🎉 終わりました：A2-27（採点し直し）。このランタイムで {N_DONE} 個。まだ残っている仕事：{len(LEFT)} 個' + (f'（{", ".join(LEFT)}）' if LEFT else '（全部そろいました）'), tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
